# Predicción de aprobación del curso

## Importar librerías necesarias

In [263]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

## Cargar el conjunto de datos

In [264]:
students_df = pd.read_excel('../data/raw/M01-M02-M03.xlsx', sheet_name='Aprobación de curso')

UMBRAL_APROBACION = 70
students_df['Aprobado'] = (students_df['Total'] >= UMBRAL_APROBACION).astype(int)

students_df.info()
students_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 38 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   school      395 non-null    str    
 1   sex         395 non-null    str    
 2   age         395 non-null    int64  
 3   address     395 non-null    str    
 4   famsize     395 non-null    str    
 5   Pstatus     395 non-null    str    
 6   Medu        395 non-null    int64  
 7   Fedu        395 non-null    int64  
 8   Mjob        395 non-null    str    
 9   Fjob        395 non-null    str    
 10  reason      395 non-null    str    
 11  guardian    395 non-null    str    
 12  traveltime  395 non-null    int64  
 13  studytime   395 non-null    int64  
 14  failures    395 non-null    int64  
 15  schoolsup   395 non-null    str    
 16  famsup      395 non-null    str    
 17  paid        395 non-null    str    
 18  activities  395 non-null    str    
 19  nursery     395 non-null    str    
 20 

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,health,absences,G1,Homework,G2,Quizzes,G3,Test,Total,Aprobado
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,3,6,5,25,6,30,6,30,28.33,0
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,3,4,5,25,5,25,6,30,26.67,0
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,3,10,7,35,8,40,10,50,41.67,0
3,GP,F,15,U,GT3,T,4,2,health,services,...,5,2,15,75,14,70,15,75,73.33,1
4,GP,F,16,U,GT3,T,3,3,other,other,...,5,4,6,30,10,50,10,50,43.33,0


## Limpieza de datos

### Columna 'sex'

In [265]:
gender_dict = {'M': 1, 'F': 2}
students_df['sex'] = students_df['sex'].replace(gender_dict).astype(int)

print(students_df['sex'].value_counts())

sex
2    208
1    187
Name: count, dtype: int64


### Columna 'school'

In [266]:
# Exploramos los valores de la columna 'school'
school_values = students_df['school'].value_counts()
print(school_values)

school
GP    349
MS     46
Name: count, dtype: int64


In [267]:
school_dict = {'GP': 1, 'MS': 2}
students_df['school'] = students_df['school'].replace(school_dict).astype(int)

print(students_df['school'].value_counts())

school
1    349
2     46
Name: count, dtype: int64


### Columna 'address'

In [268]:
# Exploramos los valores de la columna 'address'
address_values = students_df['address'].value_counts()
print(address_values)

address
U    307
R     88
Name: count, dtype: int64


In [269]:
address_dict = {'U': 1, 'R': 2}
students_df['address'] = students_df['address'].replace(address_dict).astype(int)

print(students_df['address'].value_counts())

address
1    307
2     88
Name: count, dtype: int64


### Columna 'famsize'

In [270]:
# Exploramos los valores de la columna 'famsize'
famsize_values = students_df['famsize'].value_counts()
print(famsize_values)

famsize
GT3    281
LE3    114
Name: count, dtype: int64


In [271]:
famsize_dict = {'GT3': 1, 'LE3': 2}
students_df['famsize'] = students_df['famsize'].replace(famsize_dict).astype(int)

print(students_df['famsize'].value_counts())

famsize
1    281
2    114
Name: count, dtype: int64


### Columna 'Pstatus'

In [272]:
# Exploramos los valores de la columna 'Pstatus'
Pstatus_values = students_df['Pstatus'].value_counts()
print(Pstatus_values)

Pstatus
T    354
A     41
Name: count, dtype: int64


In [273]:
Pstatus_dict = {'T': 1, 'A': 2}
students_df['Pstatus'] = students_df['Pstatus'].replace(Pstatus_dict).astype(int)

print(students_df['Pstatus'].value_counts())

Pstatus
1    354
2     41
Name: count, dtype: int64


### Columna 'Mjob' y 'Fjob'

In [274]:
# Exploramos los valores de la columna 'Mjob' y 'Fjob'
MFjob_values = students_df['Mjob'].value_counts()
print(MFjob_values)

Mjob
other       141
services    103
at_home      59
teacher      58
health       34
Name: count, dtype: int64


In [275]:
MFjob_dict = {'other': 1, 'services': 2, 'at_home': 3, 'teacher': 4, 'health': 5}
students_df['Mjob'] = students_df['Mjob'].replace(MFjob_dict).astype(int)
students_df['Fjob'] = students_df['Fjob'].replace(MFjob_dict).astype(int)

print(students_df['Mjob'].value_counts())
print(students_df['Fjob'].value_counts())

Mjob
1    141
2    103
3     59
4     58
5     34
Name: count, dtype: int64
Fjob
1    217
2    111
4     29
3     20
5     18
Name: count, dtype: int64


### Columna 'reason'

In [276]:
# Exploramos los valores de la columna 'reason'
reason_values = students_df['reason'].value_counts()
print(reason_values)

reason
course        145
home          109
reputation    105
other          36
Name: count, dtype: int64


In [277]:
reason_dict = {'course': 1, 'home': 2, 'reputation': 3, 'other': 4}
students_df['reason'] = students_df['reason'].replace(reason_dict).astype(int)

print(students_df['reason'].value_counts())

reason
1    145
2    109
3    105
4     36
Name: count, dtype: int64


### Columna 'guardian'

In [278]:
# Exploramos los valores de la columna 'guardian'
guardian_values = students_df['guardian'].value_counts()
print(guardian_values)

guardian
mother    273
father     90
other      32
Name: count, dtype: int64


In [279]:
guardian_dict = {'mother': 1, 'father': 2, 'other': 3}
students_df['guardian'] = students_df['guardian'].replace(guardian_dict).astype(int)

print(students_df['guardian'].value_counts())

guardian
1    273
2     90
3     32
Name: count, dtype: int64


### Columnas 'Yes' or 'No'

In [280]:
boolean_dict = {'yes': 1, 'no': 2}
students_df['schoolsup'] = students_df['schoolsup'].replace(boolean_dict).astype(int)
students_df['famsup'] = students_df['famsup'].replace(boolean_dict).astype(int)
students_df['paid'] = students_df['paid'].replace(boolean_dict).astype(int)
students_df['activities'] = students_df['activities'].replace(boolean_dict).astype(int)
students_df['nursery'] = students_df['nursery'].replace(boolean_dict).astype(int)
students_df['higher'] = students_df['higher'].replace(boolean_dict).astype(int)
students_df['internet'] = students_df['internet'].replace(boolean_dict).astype(int)
students_df['romantic'] = students_df['romantic'].replace(boolean_dict).astype(int)

### Columna 'age'

In [281]:
# Exploramos edades en 6 rangos para ver la distribución de los datos
age_ranges = pd.qcut(students_df['age'], q=6, duplicates='drop').value_counts()

print(age_ranges)

age
(14.999, 16.0]    186
(16.0, 17.0]       98
(17.0, 18.0]       82
(18.0, 22.0]       29
Name: count, dtype: int64


In [282]:
# Categorizamos las edades para que ningun grupo incluya más
# características que otro grupo, y así evitar sesgos en el modelo
bins = [-np.inf, 14, 16, 17, 18, 22, np.inf]
labels = [0, 1, 2, 3, 4, 5, 6, 7]

students_df['age'] = pd.cut(
  pd.to_numeric(students_df['age'], errors='coerce'),
  bins=bins,
  labels=list(range(len(bins) - 1)),
  include_lowest=True
)

students_df['age'] = students_df['age'].astype('Int64')
students_df['age'].value_counts().sort_index()

age
1    186
2     98
3     82
4     29
Name: count, dtype: Int64

In [283]:
# Visualizamos la tabla resultante
students_df.info()
students_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 38 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   school      395 non-null    int64  
 1   sex         395 non-null    int64  
 2   age         395 non-null    Int64  
 3   address     395 non-null    int64  
 4   famsize     395 non-null    int64  
 5   Pstatus     395 non-null    int64  
 6   Medu        395 non-null    int64  
 7   Fedu        395 non-null    int64  
 8   Mjob        395 non-null    int64  
 9   Fjob        395 non-null    int64  
 10  reason      395 non-null    int64  
 11  guardian    395 non-null    int64  
 12  traveltime  395 non-null    int64  
 13  studytime   395 non-null    int64  
 14  failures    395 non-null    int64  
 15  schoolsup   395 non-null    int64  
 16  famsup      395 non-null    int64  
 17  paid        395 non-null    int64  
 18  activities  395 non-null    int64  
 19  nursery     395 non-null    int64  
 20 

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,health,absences,G1,Homework,G2,Quizzes,G3,Test,Total,Aprobado
0,1,2,3,1,1,2,4,4,3,4,...,3,6,5,25,6,30,6,30,28.33,0
1,1,2,2,1,1,1,1,1,3,1,...,3,4,5,25,5,25,6,30,26.67,0
2,1,2,1,1,2,1,1,1,3,1,...,3,10,7,35,8,40,10,50,41.67,0
3,1,2,1,1,1,1,4,2,5,2,...,5,2,15,75,14,70,15,75,73.33,1
4,1,2,1,1,1,1,3,3,1,1,...,5,4,6,30,10,50,10,50,43.33,0


## Preparación de los datos

Se considera aprobado al estudiante con un `Total` de 70 o más (equivalente a 7/10).

In [284]:
y = students_df['Aprobado']
X = students_df.drop(columns=['Aprobado', 'Total'])

# Separación de los datos en subconjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
  X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Entrenamiento: {len(X_train)}   Prueba: {len(X_test)}')
print('Clases (0 = no aprobado, 1 = aprobado):')
print(y.value_counts().sort_index())

Entrenamiento: 316   Prueba: 79
Clases (0 = no aprobado, 1 = aprobado):
Aprobado
0    314
1     81
Name: count, dtype: int64


## Funciones para evaluar el modelo

In [285]:
def evaluar(y_real, y_pred):
  """Métricas de clasificación para un vector de predicciones."""
  return {
    'Accuracy': accuracy_score(y_real, y_pred),
    'Precision': precision_score(y_real, y_pred, zero_division=0),
    'Recall': recall_score(y_real, y_pred, zero_division=0),
    'F1': f1_score(y_real, y_pred, zero_division=0)
  }

resultados = {}

## Entrenamiento del modelo

### Modelo simple (Dummy Classifier)

In [286]:
base = DummyClassifier(strategy='most_frequent')
base.fit(X_train, y_train)

,"strategy strategy: {""most_frequent"", ""prior"", ""stratified"", ""uniform"", ""constant""}, default=""prior""Strategy to use to generate predictions.* ""most_frequent"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit`. The `predict_proba` method returns the matching one-hot encoded vector.* ""prior"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit` (like ""most_frequent""). ``predict_proba`` always returns the empirical class distribution of `y` also known as the empirical class prior distribution.* ""stratified"": the `predict_proba` method randomly samples one-hot vectors from a multinomial distribution parametrized by the empirical class prior probabilities. The `predict` method returns the class label which got probability one in the one-hot vector of `predict_proba`. Each sampled row of both methods is therefore independent and identically distributed.* ""uniform"": generates predictions uniformly at random from the list of unique classes observed in `y`, i.e. each class has equal probability.* ""constant"": always predicts a constant label that is provided by the user. This is useful for metrics that evaluate a non-majority class. .. versionchanged:: 0.24 The default value of `strategy` has changed to ""prior"" in version 0.24.",'most_frequent'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness to generate the predictions when``strategy='stratified'`` or ``strategy='uniform'``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"constant constant: int or str or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
Name,Type,Value
"class_prior_ class_prior_: ndarray of shape (n_classes,) or list of such arraysFrequency of each class observed in `y`. For multioutput classificationproblems, this is computed independently for each output.","ndarray[float64](2,)","[0.79,0.21]"
"classes_ classes_: ndarray of shape (n_classes,) or list of such arraysUnique class labels observed in `y`. For multi-output classificationproblems, this attribute is a list of arrays as each output has anindependent set of possible classes.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X` hasfeature names that are all strings.","ndarray[object](36,)","['school','sex','age',...,'Quizzes','G3','Test']"
n_classes_ n_classes_: int or list of intNumber of label for each output.,int,2
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,36
n_outputs_ n_outputs_: intNumber of outputs.,int,1
sparse_output_ sparse_output_: boolTrue if the array returned from predict is to be in sparse CSC format.Is automatically set to True if the input `y` is passed in sparseformat.,bool,False


In [287]:
resultados['Base'] = evaluar(y_test, base.predict(X_test))
print(resultados['Base'])

{'Accuracy': 0.7974683544303798, 'Precision': 0.0, 'Recall': 0.0, 'F1': 0.0}


### Modelo árbol de decisión (Decision Tree Classifier)

In [288]:
from sklearn.tree import DecisionTreeClassifier


decision_tree = DecisionTreeClassifier(random_state=42)
decision_tree.fit(X_train, y_train)

resultados['Decision Tree'] = evaluar(y_test, decision_tree.predict(X_test))
print(resultados['Decision Tree'])

{'Accuracy': 0.9493670886075949, 'Precision': 0.8333333333333334, 'Recall': 0.9375, 'F1': 0.8823529411764706}


### Clasificador SGD

In [289]:
from sklearn.linear_model import SGDClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

sgd = make_pipeline(
  SimpleImputer(strategy='median'),
  StandardScaler(),
  SGDClassifier(max_iter=5, tol=None, random_state=42)
)

sgd.fit(X_train, y_train)

resultados['SGD'] = evaluar(y_test, sgd.predict(X_test))
print(resultados['SGD'])

{'Accuracy': 0.9620253164556962, 'Precision': 0.8823529411764706, 'Recall': 0.9375, 'F1': 0.9090909090909091}


### Modelo de regresión logística (Logistic Regression)

In [290]:
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

logistic = make_pipeline(
  SimpleImputer(strategy='median'),
  StandardScaler(),
  LogisticRegression(max_iter=1000, random_state=42)
)

logistic.fit(X_train, y_train)

resultados['Logistic Regression'] = evaluar(y_test, logistic.predict(X_test))
print(resultados['Logistic Regression'])

{'Accuracy': 0.9493670886075949, 'Precision': 0.8333333333333334, 'Recall': 0.9375, 'F1': 0.8823529411764706}


### Modelo Random Forest

In [291]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

random_forest = make_pipeline(
  SimpleImputer(strategy='median'),
  RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
)

random_forest.fit(X_train, y_train)

resultados['Random Forest'] = evaluar(y_test, random_forest.predict(X_test))
print(resultados['Random Forest'])

{'Accuracy': 0.9493670886075949, 'Precision': 0.8333333333333334, 'Recall': 0.9375, 'F1': 0.8823529411764706}


## Comparación de modelos

Se ordenan los modelos por F1, que combina precisión y recall para la clase positiva (aprobado = 1).

In [292]:
comparacion = pd.DataFrame(resultados).T.sort_values('F1', ascending=False)
comparacion.round(4)

,Accuracy,Precision,Recall,F1
SGD,0.9620,0.8824,0.9375,0.9091
Logistic Regression,0.9494,0.8333,0.9375,0.8824
Decision Tree,0.9494,0.8333,0.9375,0.8824
Random Forest,0.9494,0.8333,0.9375,0.8824
Base,0.7975,0.0000,0.0000,0.0000


## Exportar el modelo final

In [293]:
import joblib

# Exportamos el modelo con mejor desempeño
joblib.dump(sgd, '../models/approval_sgd_model.joblib')

['../models/approval_sgd_model.joblib']